# 스케일링 법칙: 모델 성능 예측 - 스케일링 법칙의 원리

- Tutorial ID: `adv-10-1`
- Tutorial: 스케일링 법칙: 모델 성능 예측
- Section ID: `adv-10-1-1`
- Section: 스케일링 법칙의 원리


In [ ]:
# ============================================================
# 코드 읽는 법 — 스케일링 법칙의 원리
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("스케일링 법칙 (Scaling Laws)")
print("=" * 60)

In [ ]:
# 1. 멱법칙 시뮬레이션

In [ ]:
print("\n1. 멱법칙: L(N) = aN^(-α) + L_irreducible")
print("-" * 45)

# Kaplan et al. 스케일링 법칙 파라미터 (근사)
alpha_N = 0.076  # 모델 크기 지수
alpha_D = 0.095  # 데이터 크기 지수
a_N = 8.8e13 ** alpha_N
L_irreducible = 1.69  # 줄일 수 없는 손실 (데이터 엔트로피)

def loss_vs_params(N):
    """모델 파라미터 수에 따른 손실"""
    return (a_N / N ** alpha_N) + L_irreducible

print("모델 크기(파라미터) vs 손실:")
for n_params in [1e6, 1e7, 1e8, 1e9, 1e10, 1e11]:
        loss = loss_vs_params(n_params)
    name = f"{n_params:.0e}"
    bar = '█' * int((4.0 - loss) * 10)
    print(f"  {name:>8s} params: L = {loss:.3f} {bar}")

In [ ]:
# 2. Chinchilla 최적 비율

In [ ]:
print("\n2. Chinchilla 최적 비율")
print("-" * 45)

def chinchilla_optimal(compute_budget):
    """
    Chinchilla: N과 D를 같은 비율로 스케일
    C ≈ 6 * N * D (대략적)
    최적: D ≈ 20 * N
    """
    # C = 6 * N * D, D = 20 * N
    # C = 6 * N * 20 * N = 120 * N^2
    N_opt = np.sqrt(compute_budget / 120)
    D_opt = 20 * N_opt
    return N_opt, D_opt

print("계산 예산별 최적 모델 크기:")
print(f"{'계산량':>12s} {'최적 N':>12s} {'최적 D':>12s} {'비율 D/N':>10s}")
for C in [1e18, 1e19, 1e20, 1e21, 1e23]:
    N, D = chinchilla_optimal(C)
    print(f"  {C:.0e}  {N:.2e}  {D:.2e}  {D/N:.0f}x")

print("""
실제 모델 비교:
  GPT-3 175B:  300B 토큰 → D/N ≈ 1.7 (부족!)
  Chinchilla 70B: 1.4T 토큰 → D/N ≈ 20 (최적!)
  LLaMA-2 70B: 2T 토큰 → D/N ≈ 29 (과훈련)
  
→ 데이터를 더 많이 훈련하는 것이 더 효율적!
""")

In [ ]:
# 3. 계산 비용 분석

In [ ]:
print("3. 실제 훈련 비용 추정")
print("-" * 45)

def training_cost(N, D, gpu_tflops=150, gpu_price_hr=3.0):
    """훈련 비용 추정 (A100 기준)"""
    C = 6 * N * D  # 총 FLOPS
    gpu_hours = C / (gpu_tflops * 1e12 * 3600)
    cost = gpu_hours * gpu_price_hr
    return gpu_hours, cost

models = [
    ("GPT-2 Small", 124e6, 300e9),
    ("GPT-2 Large", 774e6, 300e9),
    ("LLaMA-7B", 7e9, 1e12),
    ("LLaMA-70B", 70e9, 2e12),
    ("GPT-4 (추정)", 1.8e12, 13e12),
]

print(f"{'모델':>15s} {'GPU 시간':>12s} {'비용 ($)':>12s}")
for name, N, D in models:
    hours, cost = training_cost(N, D)
    print(f"  {name:>13s}  {hours:>10,.0f}h  USD {cost:>10,.0f}")